## 注意

このノートブックは miniJSRT_database を用いたデモです。  
データセット本体は本リポジトリに含まれていません。利用する場合は，必ず公式配布元から各自で取得してください。

- miniJSRT_database: http://imgcom.jsrt.or.jp/minijsrtdb/

Segmentation01 または Segmentation02 を公式配布元からダウンロードし，各画像フォルダに対応する画像データを，各自のローカル環境または Google Colab などの実行環境に配置してください。

本ノートブックは研究・教育目的のデモであり，医療診断用途を意図したものではありません。

## 想定ディレクトリ構成

```text
SegmentationUnet/
├─ org_train/
├─ label_train/
├─ org_test/
├─ label_test/
└─ unet.ipynb

必要な画像は，上記フォルダに各自で配置してください。

# UNetを用いて肺のセグメンテーションを行う  
※次の手順でランタイムを GPU に設定してください。
1.   「ランタイムのタイプを変更 (Change runtime type)」 を選択
2. 「ハードウェア アクセラレータ (Hardware accelerator)」 を GPU に設定
3. 「保存 (Save)」 をクリック

In [ ]:
# このコードを実行することでGoogleドライブ上のファイルにノートブックからアクセスできるようになります
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 作業ディレクトリまで移動する
%cd '/content/drive/MyDrive/UOEH_UKK/SegmentationUnet'

In [ ]:
# 用いるライブラリのインポート
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [ ]:
# 学習に用いる定数の定義
# IMAGE_SIZE = 64
IMAGE_SIZE = 224
SEGMENTATION_TARGET_LABEL = 255
BATCH_SIZE = 12
EPOCHS = 50 # 学習回数
NUM_TRAIN_IMAGES = 48 # 学習データ枚数(最大199枚まで)

In [ ]:
################################################
# 画像読み込み
################################################
def load_dataset(image_dir, label_dir, imagesize, max_images=None):

    images = []
    labels = []
    filenames = []

    image_files = sorted(os.listdir(image_dir))

    count = 0

    for fn in image_files:

        # .DS_Store を除外
        if fn.startswith("."):
            continue

        base = os.path.splitext(fn)[0]

        img_path = os.path.join(image_dir, fn)
        label_path = os.path.join(label_dir, base + "_label.png")

        if not os.path.exists(label_path):
            print("label not found:", label_path)
            continue

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img,(imagesize,imagesize))

        label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
        label = cv2.resize(label,(imagesize,imagesize))

        img = np.expand_dims(img,0)
        label = np.expand_dims(label,0)

        images.append(img)
        labels.append(label)
        filenames.append(fn)

        count += 1

        if max_images is not None and count >= max_images:
            break

    return np.array(images), np.array(labels), filenames

In [ ]:
################################################
# 保存
################################################
def save_images(savepath,filenames,imagelist):

    os.makedirs(savepath,exist_ok=True)

    for i,fn in enumerate(filenames):

        filename = os.path.join(savepath,fn)

        img = imagelist[i][0]

        cv2.imwrite(filename,img)


In [ ]:
################################################
# ラベル変換
################################################
def convlabelnum2desire(label, orgnum, outnum):

    label[label != orgnum] = 0
    label[label == orgnum] = outnum

In [ ]:
################################################
# U-Net
################################################
class DoubleConv(nn.Module):

    def __init__(self,in_ch,out_ch):

        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch,out_ch,3,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self,x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = DoubleConv(1,64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(64,128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(128,256)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(256,512)
        self.pool4 = nn.MaxPool2d(2)

        self.center = DoubleConv(512,1024)

        self.up4 = nn.ConvTranspose2d(1024,512,2,stride=2)
        self.dec4 = DoubleConv(1024,512)

        self.up3 = nn.ConvTranspose2d(512,256,2,stride=2)
        self.dec3 = DoubleConv(512,256)

        self.up2 = nn.ConvTranspose2d(256,128,2,stride=2)
        self.dec2 = DoubleConv(256,128)

        self.up1 = nn.ConvTranspose2d(128,64,2,stride=2)
        self.dec1 = DoubleConv(128,64)

        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        c = self.center(self.pool4(e4))

        d4 = self.up4(c)
        d4 = torch.cat([d4,e4],dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3,e3],dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2,e2],dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1,e1],dim=1)
        d1 = self.dec1(d1)

        out = torch.sigmoid(self.out(d1))

        return out

In [ ]:
################################################
# データ読み込み
################################################
image_train, label_train, train_filenames = load_dataset(
    "./org_train/", "./label_train/", IMAGE_SIZE, NUM_TRAIN_IMAGES
)

image_test, label_test, test_filenames = load_dataset(
    "./org_test/", "./label_test/", IMAGE_SIZE
)

In [ ]:
# 白黒反転
image_train = np.max(image_train) - image_train
image_test  = np.max(image_test)  - image_test

# ラベル変換
convlabelnum2desire(label_train,SEGMENTATION_TARGET_LABEL,255)
convlabelnum2desire(label_test,SEGMENTATION_TARGET_LABEL,255)

# 正規化
image_train = image_train.astype(np.float32)
label_train = label_train.astype(np.float32)
image_test  = image_test.astype(np.float32)
label_test  = label_test.astype(np.float32)
image_train /= np.max(image_train)
label_train /= np.max(label_train)
image_test /= np.max(image_test)
label_test /= np.max(label_test)

In [ ]:
# 学習データの画像を表示する
plt.imshow(image_train[0][0], cmap="gray")
plt.title("sample train image")
plt.show()

In [ ]:
################################################
# Tensor化
################################################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_train = torch.tensor(image_train).float().to(device)
label_train = torch.tensor(label_train).float().to(device)

image_test = torch.tensor(image_test).float().to(device)
label_test = torch.tensor(label_test).float().to(device)

In [ ]:
################################################
# モデル
################################################

model = UNet().to(device)

criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
################################################
# training
################################################
train_loss_history = []

for epoch in range(EPOCHS):

    model.train()

    perm = torch.randperm(image_train.size(0))

    loss_sum = 0

    for i in range(0, image_train.size(0), BATCH_SIZE):

        idx = perm[i:i+BATCH_SIZE]

        x = image_train[idx]
        y = label_train[idx]

        optimizer.zero_grad()

        pred = model(x)

        loss = criterion(pred, y)

        loss.backward()

        optimizer.step()

        loss_sum += loss.item()

    epoch_loss = loss_sum / len(range(0, image_train.size(0), BATCH_SIZE))

    train_loss_history.append(epoch_loss)

    print("Epoch", epoch+1, "Loss", epoch_loss)

In [ ]:
################################################
# 学習過程のLossをグラフで可視化
################################################
plt.figure(figsize=(6,4))

plt.plot(train_loss_history)

plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.show()

In [ ]:
################################################
# 推定
################################################

model.eval()

with torch.no_grad():

    results = model(image_test).cpu().numpy()

In [ ]:
################################################
# 定量評価
################################################
def pixel_accuracy(pred, gt):

    correct = np.sum(pred == gt)
    total = gt.size

    return correct / total

def dice_score(pred, gt):

    smooth = 1e-6

    intersection = np.sum(pred * gt)
    union = np.sum(pred) + np.sum(gt)

    dice = (2 * intersection + smooth) / (union + smooth)

    return dice

def iou_score(pred, gt):

    smooth = 1e-6

    intersection = np.sum(pred * gt)
    union = np.sum(pred) + np.sum(gt) - intersection

    iou = (intersection + smooth) / (union + smooth)

    return iou

dice_list = []
iou_list = []
acc_list = []
test_loss_list = []

for i in range(len(results)):

    pred_prob = results[i][0]
    pred = (pred_prob > 0.5).astype(np.float32)

    gt = label_test[i][0].cpu().numpy()
    gt = (gt > 0.5).astype(np.float32)

    # Dice
    dice = dice_score(pred, gt)

    # IoU
    iou = iou_score(pred, gt)

    # Accuracy
    acc = pixel_accuracy(pred, gt)

    dice_list.append(dice)
    iou_list.append(iou)
    acc_list.append(acc)

    # loss計算
    pred_tensor = torch.tensor(pred_prob).unsqueeze(0).unsqueeze(0).to(device)
    gt_tensor = torch.tensor(gt).unsqueeze(0).unsqueeze(0).to(device)

    loss = criterion(pred_tensor, gt_tensor)

    test_loss_list.append(loss.item())

print("Mean Test Loss :", np.mean(test_loss_list))
print("Mean Accuracy  :", np.mean(acc_list))
print("Mean Dice :", np.mean(dice_list))
print("Mean IoU  :", np.mean(iou_list))

In [ ]:
################################################
# 定性評価
################################################

n = 5

plt.figure(figsize=(15,9))

for i in range(n):

    # input image
    ax = plt.subplot(3,n,i+1)
    plt.imshow(image_test[i][0].cpu(), cmap="gray")
    ax.set_title("image")
    ax.axis("off")

    # label (ground truth)
    ax = plt.subplot(3,n,i+1+n)
    plt.imshow(label_test[i][0].cpu(), cmap="gray")
    ax.set_title("label")
    ax.axis("off")

    # prediction
    ax = plt.subplot(3,n,i+1+2*n)
    plt.imshow(results[i][0], cmap="gray")
    ax.set_title("prediction")
    ax.axis("off")

plt.show()

In [ ]:
################################################
# 推定結果の画像をファイル保存
################################################
results *= 255.0
save_images("./result/fat_out", test_filenames, results)

# U-Netによるセグメンテーション（まとめ）
 **目的**   
 胸部X線画像から肺領域をセグメンテーションするモデルの学習

**処理の流れ**   
1. データ読み込み
2. 前処理
3. U-Netモデル作成
4. 学習
5. 推論
6. 評価

* エポック数を増やすことで学習が進み，精度が向上する場合がある．  
* しかし，エポック数を増やしすぎると過学習が発生する可能性がある．  
* 学習に使用する画像枚数によっても結果が変化し，データ数が多いほど一般的に安定した結果が得られる．
* さらに，時間があれば学習率やloss関数などを変更して実験を行うことで，モデルの性能が  
どのように変化するかを確認することも有用である.




